In [6]:
import numpy
from PIL import Image

In [7]:
colordict = {
    "black": (0, 0, 0),
    "white": (255, 255, 255),
    "red" : (255, 0, 0),
    "green": (0, 255, 0),
    "bule": (0, 0, 255),
    "yellow": (255, 255, 0),
    "pink": (252, 94, 175),
    "orange": (255, 140, 0)
}

In [8]:
# adaptee
class PrintingPaper:
    __widthInch:float
    __heightInch:float
    __paperType:str
    def __init__(self, widthInch:float, heightInch:float, paperType:str):
        self.__widthInch = widthInch
        self.__heightInch = heightInch
        self.__paperType = paperType
    def getWidthInch(self):
        return self.__widthInch
    def getHeightInch(self):
        return self.__heightInch
    def __str__(self):
        return "Printing paper is " + self.__paperType + " paper, Width: " + str(round(self.__widthInch,2)) + " Inch, Heiht:" + str(round(self.__heightInch,2)) + " Inch"


In [9]:
# adaptee
class ClothBag:
    __widthCM: float
    __heightCM: float
    __color: tuple
    def __init__(self, widthCM:float, heightCM:float, color:tuple):
        self.__widthCM = widthCM
        self.__heightCM = heightCM
        self.__color = color
    def getWidthCM(self):
        return self.__widthCM
    def getHeightCM(self):
        return self.__heightCM
    def getColor(self):
        return self.__color
    def __str__(self):
        result = "ClothBag color is "
        haveInDict = False
        for key, value in colordict.items():
            if(numpy.array_equal(value, self.__color)):
                haveInDict = True
                break
        if haveInDict:
            result += key
        else:
            result += str(self.__color)
        result += ", Width: " + str(round(self.__widthCM,2)) + " cm, Heiht:" + str(round(self.__heightCM,2)) + " cm"
        return result

In [10]:
# interface
from abc import ABC, abstractmethod

class Layout(ABC): 
    @abstractmethod
    def getWidthPixel():
        pass
    @abstractmethod
    def getHeightPixel():
        pass

In [11]:
# adapter
class PrintingPaperAdapter(Layout):
    __paper: PrintingPaper
    def __init__(self, adaptee: PrintingPaper):
        self.__paper = adaptee
    def getWidthPixel(self):
        return self.__paper.getWidthInch()*96
    def getHeightPixel(self):
        return self.__paper.getHeightInch()*96
    def __str__(self):
        return "Printing paper layout size: " + str(self.__paper.getWidthInch()) + "x" + str(self.__paper.getHeightInch()) + " inch, " + str(self.getWidthPixel()) + "x" + str(self.getHeightPixel()) + " pixel"

    

In [12]:
# adapter
class ClothBagAdapter:
    __bag: ClothBag
    def __init__(self, adaptee:ClothBag):
        self.__bag = adaptee
    def getWidthPixel(self):
        return self.__bag.getWidthCM()*37.8
    def getHeightPixel(self):
        return self.__bag.getHeightCM()*37.8
    def __str__(self):
        return "Cloth bag layout size: " + str(self.__bag.getWidthCM()) + "x" + str(self.__bag.getHeightCM()) + " cm, " + str(self.getWidthPixel()) + "x" + str(self.getHeightPixel()) + " pixel"

    

In [13]:
# client
class Illustration:
    __name:str
    __widthPixel:int
    __heightPixel:int
    __data:list
    def __init__(self, name:str, widthPixel:int, heightPixel:int):
        self.__name = name
        self.__widthPixel = widthPixel
        self.__heightPixel = heightPixel
        self.__data = numpy.zeros((self.__heightPixel, self.__widthPixel, 3), dtype=numpy.uint8)
        self.__data.fill(255)
    def getWidthPixel(self):
        return self.__widthPixel
    def getHeightPixel(self):
        return self.__heightPixel
    def getData(self):
        return self.__data
    def getName(self):
        return self.__name
    def __str__(self):
        return "Illustration name: " + self.__name + ", Width: " + str(self.__widthPixel) + " pixel, Height: " + str(self.__heightPixel) + " pixel"
    def canPrintWith(self, layout: Layout):
        if (self.__widthPixel < layout.getWidthPixel() and self.__heightPixel < layout.getHeightPixel()):
            print("")
            return True
        else:
            return False
    

In [25]:
# main1 create Illustration
illust1 = Illustration("MySquare", 500, 500)

data = illust1.getData()

# paint illust11
data[50,50]
for i in range(50,450):
    for j in range(20):
        data[50+j,i] = colordict['red']
        data[450-j,i] = colordict['yellow']
        data[i,50+j] = colordict['green']
        data[i,450-j] = colordict['bule']
        
image = Image.fromarray(data)
image.save("MySquare.png")

img = Image.open(r"./MySquare.png")
img.show()


In [15]:
# print illustration to PrintingPaper function
def printIllustration(illust:Illustration, paper:PrintingPaper):
    adapter = PrintingPaperAdapter(paper)
    if not illust.canPrintWith(adapter):
        print("the size of illustration is not fit to this printing paper, " + str(paper))
        return
    illustData = illust.getData()
    paperData = numpy.zeros((int(adapter.getHeightPixel()), int(adapter.getWidthPixel()), 3), dtype=numpy.uint8)
    paperData.fill(255)
    for i in range(int(adapter.getWidthPixel())):
        for j in range(int(adapter.getHeightPixel())):
            if j >= illust.getHeightPixel()-1:
                break
            paperData[i,j] = illustData[i,j]
        if i >= illust.getWidthPixel()-1:
            break
        paperData[i,j] = illustData[i,j]   
    print("print the illustration with printing paper, "+ str(paper))        
    image = Image.fromarray(paperData)
    image.show()

In [16]:
# screen illustration to ClothBag function
def screenIllustration(illust:Illustration, bag:ClothBag):
    adapter = ClothBagAdapter(bag)
    if not illust.canPrintWith(adapter):
        print("the size of illustration is not fit to this cloth bag, " + str(bag))
        return
    illustData = illust.getData()
    w = int(adapter.getHeightPixel())
    h = int(adapter.getWidthPixel())
    bagData = numpy.zeros((w, h, 3), dtype=numpy.uint8)
    
    for i in range(w-1):
        for j in range(h-1):
            bagData[i][j] = bag.getColor()
            
    for i in range(int(adapter.getWidthPixel())):
        for j in range(int(adapter.getHeightPixel())):
            if j >= illust.getHeightPixel()-1:
                break
            if (numpy.array_equal(illustData[i,j],colordict['white'])):                
                continue
            bagData[i,j] = illustData[i,j]
        if i >= illust.getWidthPixel()-1:
            break
        
    print("screen the illustration with cloth bag, " + str(bag))       
    image = Image.fromarray(bagData)
    image.show()

In [27]:
# main2 PrintingPaper
poscard = PrintingPaper(4,6,"Matte")
stickerPaper = PrintingPaper(8.27,11.69,"Glossy")   # size A4

print("..illust1..")
print(illust1)

print("\n..poscard..")
print(poscard)
print(PrintingPaperAdapter(poscard))

print("\n..stickerPaper..")
print(stickerPaper)
print(PrintingPaperAdapter(stickerPaper))


..illust1..
Illustration name: MySquare, Width: 500 pixel, Height: 500 pixel

..poscard..
Printing paper is Matte paper, Width: 4 Inch, Heiht:6 Inch
Printing paper layout size: 4x6 inch, 384x576 pixel

..stickerPaper..
Printing paper is Glossy paper, Width: 8.27 Inch, Heiht:11.69 Inch
Printing paper layout size: 8.27x11.69 inch, 793.92x1122.24 pixel


In [18]:
print(illust1.canPrintWith(PrintingPaperAdapter(poscard)))
print(illust1.canPrintWith(PrintingPaperAdapter(stickerPaper)))

False

True


In [28]:
printIllustration(illust1, poscard)

the size of illustration is not fit to this printing paper, Printing paper is Matte paper, Width: 4 Inch, Heiht:6 Inch


In [29]:
printIllustration(illust1, stickerPaper)


print the illustration with printing paper, Printing paper is Glossy paper, Width: 8.27 Inch, Heiht:11.69 Inch


In [30]:
# main3 ClothBag
pinkBag = ClothBag(23, 28, colordict['pink'])
orangeBag = ClothBag(23, 28, colordict['orange'])

print("..illust1..")
print(illust1)

print("\n..pinkBag..")
print(pinkBag)
print(ClothBagAdapter(pinkBag))

print("\n..orangeBag..")
print(orangeBag)
print(ClothBagAdapter(orangeBag))

..illust1..
Illustration name: MySquare, Width: 500 pixel, Height: 500 pixel

..pinkBag..
ClothBag color is pink, Width: 23 cm, Heiht:28 cm
Cloth bag layout size: 23x28 cm, 869.4x1058.3999999999999 pixel

..orangeBag..
ClothBag color is orange, Width: 23 cm, Heiht:28 cm
Cloth bag layout size: 23x28 cm, 869.4x1058.3999999999999 pixel


In [22]:
print(illust1.canPrintWith(ClothBagAdapter(pinkBag)))
print(illust1.canPrintWith(ClothBagAdapter(orangeBag)))


True

True


In [31]:
screenIllustration(illust1, pinkBag)


screen the illustration with cloth bag, ClothBag color is pink, Width: 23 cm, Heiht:28 cm


In [32]:
screenIllustration(illust1, orangeBag)


screen the illustration with cloth bag, ClothBag color is orange, Width: 23 cm, Heiht:28 cm
